# Categorical Variables and Interactions

Add a categorical sector predictor and a training-by-sector interaction to a salary regression.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf

sns.set_theme(style="whitegrid")

from pathlib import Path


def find_repo_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists():
            return path
    raise RuntimeError("Could not find repository root")


ROOT = find_repo_root()
DATA = ROOT / "data"

In [ ]:
df = pd.read_csv(DATA / "career_outcomes.csv")
df["Sector"].value_counts()

In [ ]:
base = smf.ols(
    "salary_k_eur ~ experience_years + training_hours + ai_tool_use + C(sector)",
    data=df,
).fit()
print(base.summary())

## Reference categories

`statsmodels` chooses one sector as the reference. Other sector coefficients are read relative to that reference, holding the other predictors constant.

In [ ]:
interaction = smf.ols(
    "salary_k_eur ~ experience_years + training_hours + ai_tool_use + C(sector) + training_hours:C(sector)",
    data=df,
).fit()

In [ ]:
pd.DataFrame(
    {
        "model": ["base", "with_interaction"],
        "r_squared": [base.rsquared, interaction.rsquared],
    }
)

In [ ]:
grid = pd.DataFrame(
    {
        "training_hours": [10, 30, 50, 70] * 4,
        "sector": ["finance"] * 4 + ["public"] * 4 + ["retail"] * 4 + ["technology"] * 4,
        "experience_years": df["experience_years"].median(),
        "ai_tool_use": df["ai_tool_use"].median(),
    }
)
grid["predicted_salary"] = interaction.predict(grid)
grid

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.lineplot(data=grid, x="training_hours", y="predicted_salary", hue="sector", marker="o", ax=ax)
ax.set(title="Predicted salary by sector and training", ylabel="Predicted salary, thousand EUR")

## Interpretation

Explain one sector coefficient and the interaction relative to the reference category, and decide whether allowing sector-specific training slopes is justified.